# vfairness · Module 9 · Multi-Agent Fairness Testing

This notebook walks through every analyzer in `vfairness.multi_agent` on a
small synthetic three-agent setup. It shows:

1. **CompositionalityAnalyzer** — system bias vs aggregate component bias.
2. **EmergentBiasDetector** — amplification factor + bootstrap significance.
3. **GroupthinkDetector** — convergence trajectories + coalition detection.
4. **AdversarialCollusionDetector** — bias amplified by interaction (Khan et al. 2023).
5. **DelegationRoutingAuditor** — demographic-conditional routing (Bertrand & Mullainathan 2004).
6. **NegotiationFairnessTracker** — per-turn fairness drift (Bianchi et al. 2024).
7. **MultiAgentRunHarness** — framework-agnostic capture surface.

All analyzers operate on numeric inputs you produce yourself from a real
agent run (autogen, crewai, langgraph, raw function-calling, etc.). The
synthetic data below stands in for that.


In [1]:
import numpy as np
from vfairness.multi_agent import (
    CompositionalityAnalyzer,
    EmergentBiasDetector,
    GroupthinkDetector,
    AdversarialCollusionDetector,
    DelegationRoutingAuditor,
    NegotiationFairnessTracker,
    MultiAgentRunHarness,
)

rng = np.random.default_rng(42)
N = 200                                       # samples per scenario
groups = np.array([0]*(N//2) + [1]*(N//2))    # binary demographic
print('N =', N, '  group balance =', np.mean(groups))

<local-path> TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<vfairness>/src/vfairness/llm/scorers.py:1387: UserWarning: Using keyword-based sentiment scorer (30 words only). This is a LOW-ACCURACY placeholder. Install vaderSentiment for production-quality scoring: pip install vaderSentiment
  DEFAULT_SENTIMENT_SCORER = KeywordSentimentScorer()
<vfairness>/src/vfairness/llm/scorers.py:1397: UserWarning: Using keyword-based toxicity scorer (25 words only). This is a LOW-ACCURACY placeholder. Install alt-profanity-check for production-quality scoring: pip install alt-profanity-check
  DEFAULT_TOXICITY_SCORER = KeywordToxicityScorer()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/201 [00:00<00:00, 2331.46it/s, Materializing param=bert.embeddings.LayerNorm.bias]

Loading weights:   0%|          | 1/201 [00:00<00:00, 1201.81it/s, Materializing param=bert.embeddings.LayerNorm.bias]

...

Loading weights: 100%|█████████▉| 200/201 [00:00<00:00, 1885.61it/s, Materializing param=classifier.bias]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1885.61it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1885.61it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1881.35it/s, Materializing param=classifier.weight]

N = 200   group balance = 0.5


## 1. CompositionalityAnalyzer

Two agents look fairly clean individually (small biases), but the composed
system exhibits a much larger bias. We expect the *amplification* scenario.


In [2]:
result = CompositionalityAnalyzer().analyze(
    component_biases={"agent_a": 0.04, "agent_b": 0.05},
    system_bias=0.22,
)
print('scenario       :', result.scenario)
print('system_score   :', result.system_score)
print('divergence     :', round(result.divergence, 4))
print('component_scores:', result.component_scores)

scenario       : amplification
system_score   : 0.22
divergence     : 0.17
component_scores: {'agent_a': 0.04, 'agent_b': 0.05}


## 2. EmergentBiasDetector

Two clean components produce a biased system output. The detector should
flag this as *emergent* (amplification factor far above 1) and significant
under the bootstrap test.


In [3]:
# Two unbiased agents.
comp = {
    'agent_a': rng.normal(0, 1, N),
    'agent_b': rng.normal(0, 1, N),
}
# System output is strongly group-conditional even though components weren't.
system_out = 0.6 * (groups == 0) + rng.normal(0, 0.1, N)

result = EmergentBiasDetector().analyze(comp, system_out, groups)
print('system_bias        :', round(result.system_bias, 3))
print('max_component_bias :', round(result.max_component_bias, 3))
print('amplification      :', round(result.amplification_factor, 2), 'x')
print('is_emergent        :', result.is_emergent)
print('is_significant     :', result.is_significant, '(p =', round(result.p_value, 4), ')')

system_bias        : 0.605
max_component_bias : 0.165
amplification      : 3.67 x
is_emergent        : True
is_significant     : True (p = 0.0 )


## 3. GroupthinkDetector

Five rounds of dialogue between three agents. Round 0 they disagree; by
round 4 they have converged. We expect groupthink to be detected with a
significant Kendall-tau trend.


In [4]:
rounds = []
for t in range(5):
    # Agents start diverse, converge over time.
    spread = max(0.05, 1.0 - 0.25 * t)
    rounds.append({
        'agent_a': list(rng.normal(0.5, spread, 4)),
        'agent_b': list(rng.normal(0.5, spread, 4)),
        'agent_c': list(rng.normal(0.5, spread, 4)),
    })

result = GroupthinkDetector().analyze_convergence(rounds)
print('convergence_trajectory:', [round(x, 3) for x in result.convergence_trajectory])
print('has_groupthink        :', result.has_groupthink)
print('echo_chamber_score    :', round(result.echo_chamber_score, 3))
print('coalition_structure   :', result.coalition_structure)
print('p_value               :', round(result.p_value, 4))

convergence_trajectory: [0.077, 0.086, 0.699, 0.851, 0.994]
has_groupthink        : True
echo_chamber_score    : 0.994
coalition_structure   : [['agent_a', 'agent_b', 'agent_c']]
p_value               : 0.01


## 4. AdversarialCollusionDetector  *(new in v0.0.8+)*

Pre-interaction: both agents are unbiased. Post-interaction: both agents
have shifted to favor group 0. The detector should attribute the new bias
to the interaction itself (not to either agent).

Statistical anchor: paired sample-level permutation test (500 perms) of
the null *pre and post outputs are exchangeable per sample*.


In [5]:
pre = {
    'agent_a': rng.normal(0, 1, N),
    'agent_b': rng.normal(0, 1, N),
}
# Post-interaction: both agents have absorbed a coordinated favoritism toward group 0.
post = {
    'agent_a': rng.normal(0, 1, N) + 1.2 * (groups == 0),
    'agent_b': rng.normal(0, 1, N) + 1.2 * (groups == 0),
}

result = AdversarialCollusionDetector(n_permutations=500, random_seed=42).analyze(pre, post, groups)
print('pre_mean_bias  :', round(result.pre_mean_bias, 3))
print('post_mean_bias :', round(result.post_mean_bias, 3))
print('collusion_score:', round(result.collusion_score, 3))
print('is_collusion   :', result.is_collusion)
print('p_value        :', round(result.p_value, 4))

pre_mean_bias  : 0.172
post_mean_bias : 1.212
collusion_score: 1.04
is_collusion   : True
p_value        : 0.002


## 5. DelegationRoutingAuditor  *(new in v0.0.8+)*

Synthetic orchestrator: 80 paired equivalent tasks, but it routes group A
to the *senior* agent and group B to the *junior* agent. Fisher's exact
should reject; Cramér's V should be very large.


In [6]:
routes = ['senior'] * 40 + ['junior'] * 40
demographics = ['A'] * 40 + ['B'] * 40

result = DelegationRoutingAuditor(alpha=0.05).analyze(routes, demographics)
print('test_used        :', result.test_used)
print('p_value          :', round(result.p_value, 6))
print('cramers_v        :', round(result.cramers_v, 3))
print('odds_ratio       :', result.odds_ratio)
print('per_route_disparity:')
for r, d in result.per_route_disparity.items():
    print(f'  {r:>8}: {d:.3f}')

test_used        : fisher
p_value          : 0.0
cramers_v        : 1.0
odds_ratio       : 0.0
per_route_disparity:
    junior: 1.000
    senior: 1.000


## 6. NegotiationFairnessTracker  *(new in v0.0.8+)*

A negotiation runs for 8 turns. Group A's representative concedes more
each turn; group B's holds firm. End-of-dialogue disparity alone would
miss that this *grew* steadily — the Mann-Kendall trend test catches it.


In [7]:
group_a_turns = [100, 90, 80, 70, 60, 50, 40, 30]
group_b_turns = [100, 99, 98, 97, 96, 95, 94, 93]

result = NegotiationFairnessTracker(alpha=0.05).analyze(group_a_turns, group_b_turns)
print('per_turn_disparity :', result.per_turn_disparity)
print('trend              :', result.trend)
print('mann_kendall_tau   :', round(result.mann_kendall_tau, 3))
print('p_value            :', round(result.p_value, 6))
print('is_widening        :', result.is_widening)
print('final_minus_initial:', result.final_minus_initial)

per_turn_disparity : [0.0, -9.0, -18.0, -27.0, -36.0, -45.0, -54.0, -63.0]
trend              : widening
mann_kendall_tau   : -1.0
p_value            : 5e-05
is_widening        : True
final_minus_initial: -63.0


## 7. MultiAgentRunHarness  *(new in v0.0.8+)*

This is the glue layer between your real agent framework and the
analyzers above. Wrap your run loop, call `record_sample()` /
`record_routing()` / `record_turn()` from inside your callbacks, and let
the harness produce the input shapes each analyzer expects.

Below: we capture 60 samples + 20 routing decisions in a single `with`
block, then drive *three* analyzers off the same trace.


In [8]:
rng2 = np.random.default_rng(7)

with MultiAgentRunHarness() as h:
    for i in range(60):
        g = i % 2
        h.record_sample(
            group=g,
            component_outputs={'agent_a': float(rng2.normal()), 'agent_b': float(rng2.normal())},
            system_output=float(0.4 * (g == 0) + rng2.normal(0, 0.2)),
        )
    for i in range(20):
        # Synthetic routing: 70% chance group A goes to senior, group B to junior.
        d = 'A' if i % 2 == 0 else 'B'
        senior = (d == 'A' and rng2.random() < 0.7) or (d == 'B' and rng2.random() < 0.3)
        h.record_routing(route='senior' if senior else 'junior', demographic=d)

print('captured n_samples       :', h.trace.n_samples)
print('captured n_routes        :', len(h.trace.routing_decisions))

# Drive analyzers off the trace.
comp_out, sys_out, groups_h = h.as_emergent_inputs()
emergent = EmergentBiasDetector().analyze(comp_out, sys_out, groups_h)
print('  emergent is_emergent   :', emergent.is_emergent, '   amplification:', round(emergent.amplification_factor, 2))

comp_bias = h.as_compositionality_input()
sys_bias = h.system_bias()
comp_result = CompositionalityAnalyzer().analyze(comp_bias, sys_bias)
print('  composition scenario   :', comp_result.scenario)

routes, demos = h.as_delegation_inputs()
deleg = DelegationRoutingAuditor().analyze(routes, demos)
print('  delegation p_value     :', round(deleg.p_value, 4), '  cramers_v:', round(deleg.cramers_v, 3))

captured n_samples       : 60
captured n_routes        : 20
  emergent is_emergent   : True    amplification: 17.29
  composition scenario   : novel_emergence
  delegation p_value     : 0.0867   cramers_v: 0.5


## Recap

| # | Detector | Failure mode | Significance test |
|---|---|---|---|
| 1 | CompositionalityAnalyzer | System vs aggregate component bias | Threshold-based scenario classification |
| 2 | EmergentBiasDetector | System bias exceeds max component bias | Bootstrap CI (500 resamples) |
| 3 | GroupthinkDetector | Convergence / echo-chamber dynamics | Kendall&rsquo;s tau + 500-perm test |
| 4 | AdversarialCollusionDetector | Bias amplified by interaction | Paired sample-level permutation (500) |
| 5 | DelegationRoutingAuditor | Demographic-conditional routing | Fisher&rsquo;s exact / chi-square + Cramér&rsquo;s V |
| 6 | NegotiationFairnessTracker | Per-turn fairness drift | Mann-Kendall trend test |
| 7 | MultiAgentRunHarness | (capture surface) | &mdash; |

The harness pattern means you can run the *same* analyzers on autogen,
crewai, langgraph, or raw function-calling traces &mdash; only the
recording callbacks change.
